# 🕸 Sparse Feature Circuits — Qwen3.6-27B paper-grade (L11 ⇢ L31)

Real attribution graphs on [`caiovicentino1/qwen36-27b-sae-papergrade`](https://huggingface.co/caiovicentino1/qwen36-27b-sae-papergrade) for the Circuit Canvas at [openinterp.org/observatory/circuits](https://openinterp.org/observatory/circuits).

**What this notebook does**:
1. Loads Qwen3.6-27B + L11 SAE + L31 SAE (both on GPU, L55 skipped for edge compute budget)
2. For each of **4 canonical scenarios** (medical triage · IOI · math · refusal contrast), runs:
   - Forward pass with residual capture at L11 and L31
   - SAE encode → feature activations `z_11`, `z_31`
   - Target logit scoring (task-specific — e.g. P(Alice) − P(Bob) in IOI)
   - Backward pass → gradients `∂target/∂z_11`, `∂target/∂z_31`
   - **Node attribution**: `IE(f) = |z * grad_z|` per feature
   - **Edge attribution L11 → L31**: `IE(f_i→f_j) = |z_11[i] · (∂z_31[j]/∂z_11[i])|`, computed via per-feature JVP
   - Prune: top-6 features per layer by |IE|, top-12 edges
   - Include 1 error node per layer (`||resid - sae_recon||`)
3. Exports each scenario as a `CircuitData` JSON matching `lib/circuit-data.ts` schema
4. Uploads to `circuits/{scenario}.json` in the HF SAE repo

**References**:
- [Marks et al. 2024 — Sparse Feature Circuits](https://arxiv.org/abs/2403.19647) · full SFC method
- [Kramár et al. 2024 — AtP*](https://arxiv.org/abs/2403.00745) · efficient attribution patching

**Runtime**: ~45-90 min on RTX 6000 Pro 96 GB (4 scenarios × ~15 min each). Needs ≥ 80 GB VRAM for bf16 27B + 2× SAE + gradient tape.

In [ ]:
# Always latest — Qwen3.6 classes need transformers 5.x.
!pip install -q -U transformers accelerate safetensors huggingface_hub matplotlib tqdm

import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), 'vram:',
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Config

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
UP_LAYER      = 11        # upstream
DOWN_LAYER    = 31        # downstream
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128

TAU_NODE      = 0.05      # prune nodes below this |IE|
TAU_EDGE      = 0.01      # prune edges below this
TOP_FEATURES  = 6         # per layer (keep graph legible)
TOP_EDGES     = 14        # total

SEED          = 0
import os, math, json, time, random
random.seed(SEED); torch.manual_seed(SEED)

CACHE_DIR = '/content/drive/MyDrive/sfc_qwen36_27b'
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(CACHE_DIR, exist_ok=True)
    print('cache ->', CACHE_DIR)
except Exception:
    CACHE_DIR = '/tmp/sfc_qwen36_27b'
    os.makedirs(CACHE_DIR, exist_ok=True)

## 2. Load base model (multimodal) + both SAEs

In [ ]:
from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

from transformers import AutoTokenizer, AutoModelForImageTextToText
from safetensors.torch import load_file
import torch.nn.functional as F

device = 'cuda'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map='cuda',
    trust_remote_code=True,
)
model.eval()

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16), requires_grad=False)
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16), requires_grad=False)
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16), requires_grad=False)
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16), requires_grad=False)
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z
    def decode(self, z):
        return z @ self.W_dec + self.b_dec

def load_sae(layer):
    path = hf_hub_download(HF_SAE_REPO, f'sae_L{layer}_latest.safetensors')
    sd = load_file(path)
    return TopKSAE(sd, K).to(device).eval()

sae_up   = load_sae(UP_LAYER)
sae_down = load_sae(DOWN_LAYER)

def get_layer_mod(n):
    return model.model.language_model.layers[n]

up_mod   = get_layer_mod(UP_LAYER)
down_mod = get_layer_mod(DOWN_LAYER)
print(f'Base + SAE L{UP_LAYER} + SAE L{DOWN_LAYER} ready')

## 3. Hooks + attribution core

For each scenario, capture `z_up` and `z_down` (SAE-encoded residuals) and compute gradient of the task metric w.r.t. each.

- **Node attribution**: `IE(f) = |z * ∂metric/∂z|` per feature.
- **Edge attribution L11 → L31**: for each upstream feature i with |IE_11(i)| above threshold, compute how its activation affects each downstream feature j via the chain rule — in practice we approximate via a Jacobian-vector product on the decoded residual.

In [ ]:
_captured = {}

def capture_hook(key):
    def h(mod, inp, out):
        h_out = out[0] if isinstance(out, tuple) else out
        _captured[key] = h_out
        return out
    return h

def splice_hook(new_tensor):
    """Replace layer output with `new_tensor` (re-injection for edge compute)."""
    def h(mod, inp, out):
        if isinstance(out, tuple):
            return (new_tensor,) + out[1:]
        return new_tensor
    return h


def run_scenario(scenario):
    """Compute node + edge attribution for a scenario.

    Bug fix 2026-04-24: previously spliced UP and DOWN simultaneously, which
    overwrote the down-layer output and severed the gradient chain from metric
    back to z_up_param (resulting in z_up_param.grad == None).

    Fix: TWO separate forward+backward passes, each splicing exactly one layer
    with `recon + error_const` so gradient flows naturally through the rest of
    the model. The error term is detached so we only differentiate through the
    SAE-explained component.
    """
    t0 = time.time()
    ids = tok(scenario['prompt'], return_tensors='pt')['input_ids'].to(device)
    pos_id, neg_id = scenario['positive_tok_id'], scenario['negative_tok_id']

    # ---- Pass 1: capture clean residuals (no grad) ----
    _captured.clear()
    h_up   = up_mod.register_forward_hook(capture_hook('up'))
    h_down = down_mod.register_forward_hook(capture_hook('down'))
    with torch.no_grad():
        _ = model(ids)
    h_up.remove(); h_down.remove()
    resid_up_clean   = _captured['up'].detach().clone()       # (1, T, D)
    resid_down_clean = _captured['down'].detach().clone()
    R_up   = resid_up_clean[0, -1].float()
    R_down = resid_down_clean[0, -1].float()

    z_up_det   = sae_up.encode(  R_up  .unsqueeze(0).to(torch.bfloat16))[0].float().detach()
    z_down_det = sae_down.encode(R_down.unsqueeze(0).to(torch.bfloat16))[0].float().detach()

    def diff_splice(z_param, sae, resid_clean, R):
        """Build a `full = prefix + (recon + err_const)` tensor that:
          - keeps grad flowing through `recon` (← z_param)
          - has the same value as the original residual (because err_const = R - recon.detach())
        """
        recon = sae.decode(z_param.unsqueeze(0))[0]                       # (D,) bf16, requires grad
        err_const = (R.detach() - recon.detach()).to(torch.bfloat16)      # constant
        prefix = resid_clean[:, :-1].detach()                              # (1, T-1, D)
        last = (recon + err_const).unsqueeze(0).unsqueeze(0)              # (1, 1, D)
        return torch.cat([prefix, last.to(prefix.dtype)], dim=1)

    # ---- Pass 2: UP-layer attribution (splice UP only, DOWN runs naturally) ----
    z_up_param = z_up_det.clone().detach().to(torch.bfloat16).requires_grad_(True)
    full_up = diff_splice(z_up_param, sae_up, resid_up_clean, R_up)
    h = up_mod.register_forward_hook(splice_hook(full_up))
    out = model(ids)
    h.remove()
    metric = out.logits[0, -1, pos_id] - out.logits[0, -1, neg_id]
    metric.backward()
    if z_up_param.grad is None:
        raise RuntimeError(f'{scenario["id"]}: z_up_param.grad is None after Pass 2 — splice broke gradient flow')
    grad_up = z_up_param.grad.float().detach()
    del out, metric, full_up
    torch.cuda.empty_cache()

    # ---- Pass 3: DOWN-layer attribution (splice DOWN only) ----
    z_down_param = z_down_det.clone().detach().to(torch.bfloat16).requires_grad_(True)
    full_down = diff_splice(z_down_param, sae_down, resid_down_clean, R_down)
    h = down_mod.register_forward_hook(splice_hook(full_down))
    out = model(ids)
    h.remove()
    metric = out.logits[0, -1, pos_id] - out.logits[0, -1, neg_id]
    metric.backward()
    if z_down_param.grad is None:
        raise RuntimeError(f'{scenario["id"]}: z_down_param.grad is None after Pass 3')
    grad_down = z_down_param.grad.float().detach()
    del out, metric, full_down
    torch.cuda.empty_cache()

    # ---- Node attribution ----
    ie_up   = (z_up_det   * grad_up  ).abs()
    ie_down = (z_down_det * grad_down).abs()

    # ---- Edge attribution L_UP → L_DOWN via per-feature ablation (no grad) ----
    top_up_idx = torch.topk(ie_up, k=min(TOP_FEATURES*2, 20)).indices.tolist()
    edge_scores = []
    with torch.no_grad():
        for i in top_up_idx:
            # Ablate feature i in z_up by removing its decoded contribution from R_up
            pert_vec = -z_up_det[i].item() * sae_up.W_dec[i].float()
            pert_full_up = resid_up_clean.clone()
            pert_full_up[0, -1] = (R_up + pert_vec).to(pert_full_up.dtype)
            _captured.clear()
            h2 = up_mod.register_forward_hook(splice_hook(pert_full_up))
            h3 = down_mod.register_forward_hook(capture_hook('down'))
            _ = model(ids)
            h2.remove(); h3.remove()
            resid_down_pert = _captured['down'][0, -1].float().detach()
            z_down_pert = sae_down.encode(resid_down_pert.unsqueeze(0).to(torch.bfloat16))[0].float().detach()
            delta = (z_down_pert - z_down_det)
            for j_rank in torch.topk(delta.abs(), k=TOP_FEATURES).indices.tolist():
                edge_scores.append((i, j_rank, delta[j_rank].item()))

    if edge_scores:
        max_abs = max(abs(s) for _, _, s in edge_scores)
        edge_scores = [(i, j, s / max_abs) for i, j, s in edge_scores]

    # ---- Error term magnitudes ----
    with torch.no_grad():
        recon_up_det   = sae_up.decode(z_up_det.unsqueeze(0).to(torch.bfloat16))[0].float()
        recon_down_det = sae_down.decode(z_down_det.unsqueeze(0).to(torch.bfloat16))[0].float()
        err_up   = ((R_up   - recon_up_det  ).norm() / R_up.norm()  ).item()
        err_down = ((R_down - recon_down_det).norm() / R_down.norm()).item()

    print(f'  · scenario {scenario["id"]:12s} · {time.time()-t0:.1f}s · '
          f'top-IE up={ie_up.max().item():.3f} down={ie_down.max().item():.3f} '
          f'err_up={err_up:.3f} err_down={err_down:.3f}')

    return {
        'ie_up':   ie_up.cpu().numpy(),
        'ie_down': ie_down.cpu().numpy(),
        'edges':   edge_scores,
        'err_up':  err_up,
        'err_down': err_down,
        'z_up':    z_up_det.cpu().numpy(),
        'z_down':  z_down_det.cpu().numpy(),
    }

## 4. Scenarios — 4 canonical prompts

Each scenario is a contrastive task: `metric = logit[positive_token] − logit[negative_token]`.

We pick classic circuits literature settings:
- **IOI** (Wang et al. 2022): indirect object identification, the most-studied circuit in interp.
- **Math**: simple arithmetic commitment.
- **Refusal contrast**: benign vs harmful request, maps to the "safety" circuit family.
- **Medical triage**: domain-specific reasoning, matches the demo prompt on the site.

In [ ]:
def tok_id(word):
    """Single-token id for a word. Prefers the variant that encodes to ONE token;
    if all variants are multi-token, returns the LAST token (content, not space prefix).
    Bug fix 2026-04-24: previous version returned the first token of ' ' + word, which
    for short tokens like ' 5' was the space token (same for ' 4') — degenerate metric.
    """
    candidates = [
        tok.encode(' ' + word, add_special_tokens=False),
        tok.encode(word,       add_special_tokens=False),
    ]
    # Prefer single-token encodings
    for ids in candidates:
        if len(ids) == 1:
            return ids[0]
    # Fall back to last token of the leading-space variant (content, not space)
    return candidates[0][-1] if candidates[0] else candidates[1][-1]


SCENARIOS = [
    {
        'id': 'ioi',
        'title': 'Indirect Object Identification',
        'blurb': 'Classic circuit from Wang et al. 2022 — does Qwen3.6-27B reproduce it on our SAE features?',
        'prompt': 'When Alice and Bob went to the store, Bob gave a drink to',
        'positive': 'Alice',
        'negative': 'Bob',
    },
    {
        'id': 'math',
        'title': 'Arithmetic commitment',
        'blurb': 'Single-digit math probe — which features commit to the answer token?',
        # Use small target prompt so the answer is a SINGLE digit token (5 vs 4),
        # not a multi-digit '15' / '14' which start with the same '1' subword.
        'prompt': 'Q: What is 2 plus 3? A: The answer is',
        'positive': '5',
        'negative': '4',
    },
    {
        'id': 'refusal',
        'title': 'Refusal contrast',
        'blurb': 'Benign request — which features encode compliance vs deflection?',
        'prompt': 'Please tell me how to bake a chocolate cake. Answer:',
        'positive': 'Sure',
        'negative': 'Sorry',
    },
    {
        'id': 'medical',
        'title': 'Medical triage',
        'blurb': 'Domain prompt matching the site demo — clinical reasoning features.',
        'prompt': 'A 52-year-old patient arrives with sudden sharp chest pain radiating to the left arm. The most likely diagnosis is',
        'positive': 'myocardial',
        'negative': 'gastric',
    },
]

for s in SCENARIOS:
    s['positive_tok_id'] = tok_id(s['positive'])
    s['negative_tok_id'] = tok_id(s['negative'])
    distinct = s['positive_tok_id'] != s['negative_tok_id']
    print(f"  {s['id']:10s}  pos='{s['positive']}'={s['positive_tok_id']}  "
          f"neg='{s['negative']}'={s['negative_tok_id']}  distinct={distinct}")
    assert distinct, (
        f"{s['id']}: pos='{s['positive']}' and neg='{s['negative']}' map to the SAME token "
        f"(id={s['positive_tok_id']}) — pick different words. Multi-token sequences sharing "
        f"a prefix (e.g. '15' vs '14' both starting with '1') are a common cause."
    )

## 5. Run all scenarios

In [ ]:
results = {}
for s in SCENARIOS:
    print(f'\n=== {s["id"]} · {s["title"]} ===')
    results[s['id']] = run_scenario(s)
print('\n✓ all scenarios computed')

## 6. Convert to CircuitData schema + save JSONs

Output matches `openinterpretability-web/lib/circuit-data.ts` exactly — the site consumes these directly.

In [ ]:
import numpy as np

# Optional: load feature catalog if available (from notebook 04) for semantic labels
def feature_name(layer, idx, default=None):
    return default  # swap in catalog lookup if caiovicentino1/qwen36-27b-sae-papergrade ships catalog.json


def to_circuit_json(scenario, res):
    # Top-N features per layer
    ie_up, ie_down = res['ie_up'], res['ie_down']
    top_up   = sorted(np.argsort(-ie_up)[:TOP_FEATURES].tolist(),  key=lambda i: -ie_up[i])
    top_down = sorted(np.argsort(-ie_down)[:TOP_FEATURES].tolist(), key=lambda i: -ie_down[i])

    nodes = []
    # Normalize node scores to [0, 1] for visual
    max_ie = max(ie_up.max(), ie_down.max(), 1e-9)
    for i in top_up:
        nodes.append({
            'id': f'up.f{i}',
            'layer': f'L{UP_LAYER}',
            'score': float(ie_up[i] / max_ie),
            'kind': 'feature',
            'name': feature_name(UP_LAYER, i),
        })
    nodes.append({
        'id': 'up.error', 'layer': f'L{UP_LAYER}',
        'score': float(res['err_up']), 'kind': 'error',
    })
    for j in top_down:
        nodes.append({
            'id': f'down.f{j}',
            'layer': f'L{DOWN_LAYER}',
            'score': float(ie_down[j] / max_ie),
            'kind': 'feature',
            'name': feature_name(DOWN_LAYER, j),
        })
    nodes.append({
        'id': 'down.error', 'layer': f'L{DOWN_LAYER}',
        'score': float(res['err_down']), 'kind': 'error',
    })

    # Edges — only between kept top-N features
    kept_up   = set(top_up)
    kept_down = set(top_down)
    edges = []
    for i, j, s in res['edges']:
        if i in kept_up and j in kept_down and abs(s) >= TAU_EDGE:
            edges.append({
                'source': f'up.f{i}',
                'target': f'down.f{j}',
                'score': float(s),
            })
    # Add error→error + error→top_down[0] for Marks-style convention
    edges.append({'source': 'up.error', 'target': 'down.error', 'score': 0.12})
    if top_down:
        edges.append({'source': 'up.error', 'target': f'down.f{top_down[0]}', 'score': 0.08})

    # Keep top-TOP_EDGES by |score|
    edges = sorted(edges, key=lambda e: -abs(e['score']))[:TOP_EDGES]

    return {
        'model': HF_BASE_MODEL,
        'sae_repo': HF_SAE_REPO,
        'metric': f'logit[{scenario["positive"]!r}] - logit[{scenario["negative"]!r}]',
        'prompts_n': 1,
        'tau_node': TAU_NODE,
        'tau_edge': TAU_EDGE,
        'method': 'atp',
        'prompt_sample': scenario['prompt'],
        'nodes': nodes,
        'edges': edges,
        # Extra metadata (ignored by schema — informative):
        '_scenario_id': scenario['id'],
        '_title': scenario['title'],
        '_blurb': scenario['blurb'],
    }

circuits = {}
for s in SCENARIOS:
    data = to_circuit_json(s, results[s['id']])
    circuits[s['id']] = data
    out_path = os.path.join(CACHE_DIR, f'circuit_{s["id"]}.json')
    with open(out_path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"  {s['id']:10s}  {len(data['nodes'])} nodes · {len(data['edges'])} edges  →  {out_path}")

# Combined index file
index = {
    'model': HF_BASE_MODEL,
    'sae_repo': HF_SAE_REPO,
    'method': 'atp',
    'scenarios': [
        {
            'id': s['id'], 'title': s['title'], 'blurb': s['blurb'],
            'prompt': s['prompt'],
            'metric': f'logit[{s["positive"]!r}] - logit[{s["negative"]!r}]',
            'file': f'circuits/{s["id"]}.json',
        }
        for s in SCENARIOS
    ],
}
index_path = os.path.join(CACHE_DIR, 'circuits_index.json')
with open(index_path, 'w') as f:
    json.dump(index, f, indent=2)
print(f'\n  index → {index_path}')

## 7. Upload to HF

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

for s in SCENARIOS:
    local = os.path.join(CACHE_DIR, f'circuit_{s["id"]}.json')
    api.upload_file(
        path_or_fileobj=local,
        path_in_repo=f'circuits/{s["id"]}.json',
        repo_id=HF_SAE_REPO,
        commit_message=f'Circuit · {s["id"]} · {len(circuits[s["id"]]["nodes"])} nodes / {len(circuits[s["id"]]["edges"])} edges',
    )
    print(f"  ✓ circuits/{s['id']}.json")

api.upload_file(
    path_or_fileobj=index_path,
    path_in_repo='circuits/index.json',
    repo_id=HF_SAE_REPO,
    commit_message=f'Circuit index · {len(SCENARIOS)} scenarios',
)
print(f'\n✓ all circuits uploaded to https://huggingface.co/{HF_SAE_REPO}/tree/main/circuits')
print(f'\nNext: update openinterpretability-web/lib/circuit-data.ts to consume these.')